# NB_Econ: charging-electricity emissions of the iron loop, and the coating break-even (Figs. 3 and 4)
Electricity-only screen: it counts the emissions attributable to the electricity that charges the loop. Construction, consumables and transport of the loop itself are not included, and no source we found quantifies them. Grid values are our own estimate from the 2023 generation mix weighted by IPCC median lifecycle intensities, not published lifecycle values. Run all cells.

In [ ]:
# Lifecycle median g CO2e/kWh (IPCC AR5 WGIII Annex III, Table A.III.2) and the part from burning fuel (direct)
LC  = {'coal':820,'gas':490,'nuclear':12,'hydro':24,'biomass':230,'wind':11,'solar':48,'geothermal':38}
DIR = {'coal':760,'gas':370}                     # all other sources: 0 direct
# EPA eGRID 2023 generation shares
MIX = {'U.S.':{'coal':0.161,'oil':0.005,'gas':0.432,'nuclear':0.185,'hydro':0.057,'biomass':0.011,'wind':0.1,'solar':0.039,'geothermal':0.004,'other fossil':0.005,'other unknown or purchased fuel':0.001},
 'Ohio':{'coal':0.236,'oil':0.008,'gas':0.591,'nuclear':0.122,'hydro':0.004,'biomass':0.002,'wind':0.021,'solar':0.01,'geothermal':0,'other fossil':0.006,'other unknown or purchased fuel':0},
 'California':{'coal':0.001,'oil':0,'gas':0.437,'nuclear':0.082,'hydro':0.148,'biomass':0.023,'wind':0.065,'solar':0.188,'geothermal':0.052,'other fossil':0.007,'other unknown or purchased fuel':0}}
ASSUME_GAS = {'oil','other fossil','other unknown or purchased fuel'}      # assumption: counted as natural gas
ETA = (0.246,0.276); eta = sum(ETA)/2             # loop electricity-to-electricity efficiency (Neumann 2024), midpoint
print(f'efficiency midpoint {eta:.3f}: {1/eta:.2f} kWh of charging electricity per kWh delivered (range {1/ETA[1]:.2f} to {1/ETA[0]:.2f})')
def grid(name):
    tot=op=0
    for k,v in MIX[name].items():
        s='gas' if k in ASSUME_GAS else k; tot+=v*LC[s]; op+=v*DIR.get(s,0)
    return tot,op
rows={}
for n in MIX: rows[n]=grid(n)
rows['wind']=(LC['wind'],0); rows['solar PV']=(LC['solar'],0)
print(f"{'charging source':12s} {'in (g/kWh)':>11s} {'burning share':>14s} {'out (g/kWh delivered)':>22s} {'low-high':>14s}")
for n,(t,o) in rows.items():
    print(f'{n:12s} {t:11.0f} {o/t:14.2f} {t/eta:22.0f} {t/ETA[1]:8.0f}-{t/ETA[0]:.0f}')
assert round(rows['U.S.'][0])==358 and round(rows['Ohio'][0])==493 and round(rows['California'][0])==240   # paper text
print('break-even charging intensity vs coal 820:', [round(820*e) for e in ETA], ' vs gas 490:', [round(490*e) for e in ETA])

In [ ]:
import matplotlib.pyplot as plt, numpy as np
labs=['Iron loop, wind','Iron loop, solar PV','Gas, burned','Coal, burned','Iron loop, California','Iron loop, U.S.','Iron loop, Ohio']
tot=[LC['wind']/eta,LC['solar']/eta,490,820]+[rows[k][0]/eta for k in ('California','U.S.','Ohio')]
fig,ax=plt.subplots(figsize=(4.5,3)); ax.barh(range(len(tot))[::-1],tot,color=['#0077BB']*2+['#8c5a2b','#EE7733']+['#0077BB']*3)
ax.set_yticks(range(len(tot))[::-1]); ax.set_yticklabels(labs); ax.set_xlabel('g CO2e per kWh delivered (electricity only)')
for i,t in enumerate(tot): ax.text(t+20,len(tot)-1-i,f'{t:,.0f}',va='center')
plt.tight_layout(); fig.savefig('fig_emissions.png',dpi=200); plt.show()

## Coating break-even (Fig. 4)
A coating of life L_c avoids, over its life, L_c/L − L_c/(fL) = (L_c/L)(1 − 1/f) leaks, where L is the years between leaks without a coating and f is the assumed factor by which the coating stretches that interval. It pays if its cost, as a fraction of one leak, is below that. L and f are **assumptions**: coupon tests in this proposal measure loss, heat transfer and integrity, not plant leak frequency.

In [ ]:
r_max = lambda Lc,L,f: (Lc/L)*(1-1/f)
print('L=6 yr, f=2: 6-yr coating', r_max(6,6,2), ' 2-yr coating', round(r_max(2,6,2),4))     # half a leak; one sixth
assert r_max(6,6,2)==0.5 and abs(r_max(2,6,2)-1/6)<1e-12
Lc=np.linspace(0.2,14,100); fig,ax=plt.subplots(figsize=(3.5,3.2))
for L,ls in [(3,'-'),(6,'--'),(12,':')]: ax.plot(Lc,r_max(Lc,L,2),ls,label=f'{L} years')
ax.axvspan(0,3,color='#ddd'); ax.axvline(6,color='#444',ls='--',lw=.8)
ax.set_xlabel('Coating life (years)'); ax.set_ylabel('Coating cost / cost of one leak'); ax.legend(title='Without a coating, a leak every:'); ax.set_ylim(0,1.05)
fig.savefig('fig_breakeven.png',dpi=200,bbox_inches='tight'); plt.show()